In [1]:
%load_ext autoreload
%autoreload 2



In [2]:
import os
import multiprocessing

from qdots_qll.utils.povms import sigmas_povm
import jax
import numpy as np
import jax.numpy as jnp
import qutip as qt

In [3]:
# os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(
#     multiprocessing.cpu_count()
# )


import tomllib
from qdots_qll.exp_design import (
    RandomExpDesign,
    MaxDetFimExpDesign,
    MaxTraceFimExpDesign,
)
from qdots_qll.run import Run, initial_run_from_config
from qdots_qll.smc import SMCUpdater, SMC_run
from qdots_qll.resamplers import LWResampler
from qdots_qll.stop_conditions import TerminationChecker

from qdots_qll.distributions import (
    est_cov,
    est_mean,
    initialize_particle_locations,
    initialize_weights,
)
from qdots_qll.models.models_scratch_for_drafting import (
    SingleQDot3Params,
)
import joblib
import logging
from datetime import datetime
from qdots_qll.utils.generate_initial_state import max_entangled_dm_vec
from pprint import pformat, pprint
import argparse
from jax_tqdm import loop_tqdm
import matplotlib.pyplot as plt

In [4]:
with open("jobs/one_dot.toml", "rb") as f:
    config = tomllib.load(f)

In [5]:
config.keys()

In [6]:
config["logging"]

In [7]:
logging.info(pformat(config["run"]))


number_of_runs = config["run"]["number_of_runs"]
number_of_runs_compilation = config["run_for_compilation"]["number_of_runs"]

print(number_of_runs)
print(number_of_runs_compilation)


ground_state_qdot = jnp.array(qt.ket2dm(qt.basis(2, 0))).flatten()
model = SingleQDot3Params(POVM_array=jnp.array(sigmas_povm))


seed = config["run"]["seed"]


key = jax.random.PRNGKey(seed=seed)


key, subkey = jax.random.split(key)


# print(initial_runs_compilation)
# keys = jax.random.split(key, number_of_runs)
true_pars = jnp.array(config["run"]["true_parameters"])


exp_design_dict = {
    "random": RandomExpDesign(0.01, 40),
    "maxdetfim": MaxDetFimExpDesign(0.01, 40, 20, lr=0.5),
    "maxtracefim": MaxTraceFimExpDesign(0.01, 40, 20, lr=0.5),
}

exp_design = exp_design_dict[config["run"]["exp_design"]]


resampler = LWResampler()


smcupdater = SMCUpdater(
    model=model,
    exp_design=exp_design,
    resampler=resampler,
    initial_state=ground_state_qdot,
    true_pars=true_pars,
    number_exp_repetitions=1,
)

In [8]:
smcupdater

In [9]:
key, subkey = jax.random.split(key)

In [10]:
run_0 = (
    lambda key: initial_run_from_config(
        key,
        model,
        config["run"],
    )
)(subkey)


stopper = TerminationChecker(config["run"]["max_iterations"])

In [11]:
keys = jax.random.split(subkey, number_of_runs)

In [12]:
keys

In [13]:
initial_runs = (
    jax.vmap(
        lambda key: initial_run_from_config(
            key,
            model,
            config["run"],
        )
    )
)(keys)

In [20]:
config["run"]["max_iterations"]

In [25]:
import matplotlib.pyplot as plt

In [22]:
n = config["run"]["max_iterations"]


@loop_tqdm(n)
def f_fori(i, r_obj):
    r_obj = smcupdater.step(r_obj)
    return r_obj


result = jax.vmap(lambda run_0: jax.lax.fori_loop(0, n, f_fori, run_0))(
    initial_runs
)

In [14]:
# result = jax.lax.fori_loop(0, n, f_fori, run_0)

In [23]:
result

In [45]:
import equinox as eqx

In [46]:
find_last_iter = lambda obj: obj.cov_array[obj.iteration]

In [53]:
result.cov_array[:, 0 : result.iteration]

In [48]:
find_last_iter(result).shape

In [34]:
a = result.cov_array[:, 1:]
a.shape
dims = a.shape[-2:]
dims

In [36]:
np.array(a).reshape(-1, *dims).shape

In [44]:
from scipy.stats import binned_statistic


def get_binned_results_from_runs(
    times_array,
    cov_array,
    bin_step,
):
    dims = cov_array[:, 1:].shape[-2:]
    cum_times = np.array(times_array[:, 1:]).cumsum(axis=1)
    cum_times_flatten = cum_times.flatten()
    cov_flatten = np.array(cov_array[:, 1:]).reshape(-1, *dims)
    bins = np.arange(
        cum_times_flatten.min(), cum_times_flatten.max() + 1, bin_step
    )

    mean_list = []
    std_list = []

    for i in range(dims[0]):
        mean_row_list = []
        std_row_list = []

        for j in range(dims[1]):

            mean_row_list.append(
                binned_statistic(
                    cum_times_flatten,
                    cov_flatten[:, i, j],
                    statistic="mean",
                    bins=bins,
                )[0]
            )
            std_row_list.append(
                binned_statistic(
                    cum_times_flatten,
                    cov_flatten[:, i, j],
                    statistic="std",
                    bins=bins,
                )[0]
            )
        mean_list.append(mean_row_list)
        std_list.append(std_row_list)
    mean_list = np.array(mean_list).transpose(2, 1, 0)
    std_list = np.array(std_list).transpose(2, 1, 0)

    cum_times_binned, _, _ = binned_statistic(
        cum_times_flatten, cum_times_flatten, statistic="mean", bins=bins
    )
    return cum_times_binned, mean_list, std_list

In [56]:
cumtimes_binned, means_binned, std_binned = get_binned_results_from_runs(result.times_array, result.cov_array, int(25))

In [59]:
for i in range(3):
    plt.plot(cumtimes_binned, means_binned[:, i, i], '-.', ms=0.9)

plt.loglog()

In [60]:
import joblib

joblib.dump(result, "firt_results3_params.job")

In [ ]:
joblib.save("")

In [26]:
plt.hist(np.array(result.times_array.flatten()), bins=100)
plt.show()

In [27]:
cov_mean = jnp.mean(result.cov_array, axis=0)

In [53]:
for i in range(3):
    plt.plot(cov_mean[:, i, i], ".", ms=0.8)

plt.loglog()